## Imports

In [2]:
import sys
from pathlib import Path
project_root = Path().absolute().parent
sys.path.append(str(project_root))

from src.retriever import hybrid_search

In [ ]:
from typing import List, Dict, Tuple, Optional
from typing import Union
from huggingface_hub import InferenceClient
from pymilvus import Function, FunctionType, MilvusClient
from pymilvus import AnnSearchRequest
from sentence_transformers import SentenceTransformer
from openai import OpenAI

from deepeval.metrics import (
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
)
from deepeval.models import LocalModel
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
import pandas as pd
from tqdm import tqdm

## Оценка RAG

### Готовим pipeline class

In [7]:
class RAG:
    def __init__(
        self,
        milvus_client: MilvusClient,
        embedding_model: SentenceTransformer,
        ranker: Function,
        openai_client: OpenAI,
        collection_name: str,
        model_name: str = "qwen/qwen3-4b",
        emb_field: str = "embedding",
        sparse_field: str = "sparse",
        emb_metric: str = "IP"
    ):
        self.milvus_client = milvus_client
        self.embedding_model = embedding_model
        self.ranker = ranker
        self.openai_client = openai_client
        self.collection_name = collection_name
        self.model_name = model_name
        self.emb_field = emb_field
        self.sparse_field = sparse_field
        self.emb_metric = emb_metric

        self.PROMPT_TEMPLATE = """
Используй контекст для ответа на вопрос.

При ответе обязательно указывай:
- ТОЧНОЕ название документа/файла;
- номер страницы, если информация найдена в контексте.

<context>
{context}
</context>

<question>
{question}
</question>
"""

    def _emb_text(self, text: str) -> List[float]:
        emb = self.embedding_model.encode(text, normalize_embeddings=True)
        return emb.tolist()

    def hybrid_search(
        self,
        question: str,
        limit: int = 10,
        dense_limit: int = 10,
        sparse_limit: int = 10,
        output_fields: Optional[List[str]] = None
    ) -> List[Dict]:
        if output_fields is None:
            output_fields = ["text", "document_name", "page_number"]
            
        dense_req = AnnSearchRequest(
            data=[self._emb_text(question)],
            anns_field=self.emb_field,
            param={"metric_type": self.emb_metric, "params": {}},
            limit=dense_limit
        )

        bm25_req = AnnSearchRequest(
            data=[question],
            anns_field=self.sparse_field,
            param={"metric_type": "BM25", "params": {}},
            limit=sparse_limit
        )

        hybrid_results = self.milvus_client.hybrid_search(
            self.collection_name,
            [dense_req, bm25_req],
            ranker=self.ranker,
            limit=limit,
            output_fields=output_fields
        )

        return [
            {
                "text": hit["entity"]["text"],
                "document_name": hit["entity"]["document_name"],
                "page_number": hit["entity"]["page_number"],
                "score": hit["distance"]
            }
            for hit in hybrid_results[0]
        ]

    def build_context(self, chunks: List[Dict]) -> str:
        context_parts = []
        for chunk in chunks:
            context_parts.append(
                f"""
        Источник: {chunk['document_name']}
        Страница: {chunk['page_number']}

        {chunk['text']}
        """
            )
        return "\n\n".join(context_parts)

    def _call_llm(
        self, 
        prompt: str, 
        max_tokens: int = 2048, 
        temperature: float = 0.5
    ) -> str:
        try:
            response = self.openai_client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                max_tokens=max_tokens,
                temperature=temperature
            )
            
            answer = response.choices[0].message.content
            
            if not answer or answer.strip() == "":
                return "МОДЕЛЬ ВЕРНУЛА ПУСТОЙ ОТВЕТ"
            
            return answer
        
        except Exception as e:
            return f"Ошибка при обращении к LLM: {str(e)}"

    def answer(
        self, 
        question: str, 
        return_retrieved: bool = False,
        search_type: str = "hybrid",
        limit: int = 10,
        max_tokens: int = 2048,
        temperature: float = 0.5
    ) -> Union[str, tuple]:
        if search_type == "embedding":
            retrieved = self.embedding_search(question, limit=limit)
        elif search_type == "bm25":
            retrieved = self.bm25_search(question, limit=limit)
        else:
            retrieved = self.hybrid_search(question, limit=limit)
        
        context = self.build_context(retrieved)
        prompt = self.PROMPT_TEMPLATE.format(context=context, question=question)
        result = self._call_llm(prompt, max_tokens=max_tokens, temperature=temperature)
        
        if return_retrieved:
            return result, retrieved
        return result

In [8]:
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-small"
COLLECTION_NAME = "UZConstitution_RAG"
MODEL_NAME = "qwen/qwen3-4b"

In [9]:
milvus_client = MilvusClient(uri="http://localhost:19530")

ranker = Function(
    name="rrf",
    input_field_names=[],
    function_type=FunctionType.RERANK,
    params={
        "reranker": "rrf", 
        "k": 50
    }
)

In [ ]:
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    cache_folder=r""
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1924.99it/s]


In [11]:
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

In [12]:
rag = RAG(
        milvus_client=milvus_client,
        embedding_model=embedding_model,
        ranker=ranker,
        openai_client=client,
        collection_name=COLLECTION_NAME,
        model_name=MODEL_NAME
    )

In [13]:
question = 'Кто на референдуме выступает от имени народа'
answer = rag.answer(question)
print(answer)

**Точное название документа/файла:** Constitution_uz_ru-1.pdf  
**Номер страницы:** 2  

Согласно статье 10 Конституции Республики Узбекистан (стр. 2), от имени народа Узбекистана могут выступать только избранные им Олий Мажлис и Президент Республики Узбекистан.


In [14]:
answer, chunks = rag.answer(question, return_retrieved=True)
print(f"Ответ: {answer}")
print(f"Найдено чанков: {len(chunks)}")

Ответ: **Точное название документа/файла:** constitution_uz_ru-1.pdf  
**Номер страницы:** 2  

Согласно статье 10 Конституции Республики Узбекистан, на референдуме могут выступать только избранные им Олий Мажлис и Президент Республики Узбекистан.
Найдено чанков: 10


In [15]:
user_input_list = [
    "Каков максимальный срок задержания человека без суда",
    "Какова роль Национальных институтов по правам человека в стране",
    "Какой деятельностью могут заниматься депутаты и члены сената в узбекистане?",
]
reference_list = [
    "Без решения суда лицо не может быть задержано на срок более сорока восьми часов.",
    "Национальные институты по правам человека дополняют существующие формы и средства защиты прав и свобод человека, содействуют развитию гражданского общества и повышению культуры прав человека.",
    "Депутаты Законодательной палаты и члены Сената, работающие в Сенате на постоянной основе, на период своих полномочий не могут заниматься другими видами оплачиваемой деятельности, кроме научной, творческой и педагогической.",
]

retrieved_contexts_list = []
response_list = []

### Отвечаем и сохраняем в Pandas df

In [19]:
def safe_answer(rag, question, retries: int = 3):
    last_error = None

    for _ in range(retries):
        try:
            return rag.answer(question, return_retrieved=True)
        except Exception as e:
            last_error = e
            continue

    print(f"[ERROR] Failed after retries: {question}")
    print(last_error)
    return "", []


rows = []

for user_input, reference in tqdm(
    zip(user_input_list, reference_list),
    total=len(user_input_list),
    desc="Отвечаю на вопросы"
):
    response, retrieved_context = safe_answer(rag, user_input)

    rows.append({
        "user_input": user_input,
        "retrieved_contexts": [
            {
                "text": c["text"],
                "document_name": c["document_name"],
                "page_number": c["page_number"],
                "score": c["score"]
            }
            for c in retrieved_context
        ],
        "response": response,
        "reference": reference
    })

df = pd.DataFrame(rows)
df.head()

Отвечаю на вопросы: 100%|██████████| 3/3 [00:11<00:00,  3.96s/it]


,user_input,retrieved_contexts,response,reference
0,Каков максимальный срок задержания человека бе...,[{'text': '## **Статья 27.** Каждый имеет пр...,**Название документа/файла:** constitution_uz_...,Без решения суда лицо не может быть задержано ...
1,Какова роль Национальных институтов по правам ...,[{'text': '## **Статья 56.** Национальные ин...,**Точное название документа/файла:** constitut...,Национальные институты по правам человека допо...
2,Какой деятельностью могут заниматься депутаты ...,[{'text': '## **Статья 104.** Депутатам Зако...,**Точное название документа/файла:** constitut...,Депутаты Законодательной палаты и члены Сената...


### Оценка через Deepeval

In [26]:
llm_evaluator = LocalModel(
    model="qwen/qwen3-4b",
    base_url="http://localhost:1234/v1/",
    api_key="lm-studio",
    temperature=0
)

### Оценка метрик retrieval

In [27]:
contextual_precision = ContextualPrecisionMetric(model=llm_evaluator)
contextual_recall = ContextualRecallMetric(model=llm_evaluator)
contextual_relevancy = ContextualRelevancyMetric(model=llm_evaluator)

test_cases = []

for index, row in df.iterrows():
    retrieval_context = [c["text"] for c in row["retrieved_contexts"]]
    
    test_case = LLMTestCase(
        input=row["user_input"],
        actual_output=row["response"],
        expected_output=row["reference"],
        retrieval_context=retrieval_context,
    )
    test_cases.append(test_case)

In [28]:
result = evaluate(
    test_cases=test_cases,
    metrics=[contextual_precision, contextual_recall, contextual_relevancy]
)

✨ You're running DeepEval's latest Contextual Precision Metric! (using qwen/qwen3-4b (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using qwen/qwen3-4b (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using qwen/qwen3-4b (Local Model), strict=False, 
async_mode=True)...

c:\AMORE\Development\Python Projects\yet-another-RAG-assistant\.venv\Lib\site-packages\rich\live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

DeepEvalError: Evaluation LLM outputted an invalid JSON. Please use a better evaluation model.